# Roastmate — EXP-ROAST-002: Model Roast (YOLOv8n-Cls) di atas DS003 Ter-dedup

Klasifikasi tingkat roasting: **Green · Light · Medium · Dark**

## Ringkasan
- `EXP-ROAST-001` **INVALID**: seluruh folder `test/` DS003 adalah duplikat persis
  (MD5) dari `train/`, sehingga skor val/test 1.0 hanyalah artefak kebocoran data.
- Eksperimen ini memakai DS003 yang sudah di-dedup (1.194 gambar unik) dengan split
  **grup-aware** dari `split.csv` (sumber kebenaran split, tidak diacak ulang di sini).
- Kalibrasi INT8 hanya memakai gambar dari split `train`.

## Aturan yang dipegang
1. Test tidak dipakai untuk training maupun kalibrasi.
2. Test dibuka **sekali** di akhir.
3. Split tidak diacak ulang di notebook — selalu baca `split.csv`.
4. Augmentasi tidak menggeser hue/saturasi ekstrem (warna = fitur roast).


## 1. Setup


In [ ]:
!pip install -q ultralytics

import hashlib, json, os, shutil, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import ultralytics

print("python     :", sys.version.split()[0])
print("ultralytics:", ultralytics.__version__)
print("torch      :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))
ultralytics.checks()


## 2. Dataset (privat, hasil dedup)

Dataset `roastmate-ds003-dedup` harus sudah ter-attach lewat `datasetDataSources`.
Isinya: `ds003_dedup/{train,val,test}/{kelas}`, `calib/`, `split.csv`, `calib_manifest.csv`.


In [ ]:
INPUT = Path("/kaggle/input")
cands = sorted({p for p in INPUT.rglob("ds003_dedup") if p.is_dir()})
assert cands, "Folder 'ds003_dedup' tidak ditemukan di /kaggle/input — pastikan dataset roastmate-ds003-dedup ter-attach."

DATA_ROOT = cands[0]
DS_DIR = DATA_ROOT.parent
CALIB = DS_DIR / "calib"
SPLIT_CSV = DS_DIR / "split.csv"
CALIB_MANIFEST = DS_DIR / "calib_manifest.csv"

print("DATA_ROOT      :", DATA_ROOT)
print("CALIB          :", CALIB, "| ada:", CALIB.is_dir())
print("SPLIT_CSV      :", SPLIT_CSV, "| ada:", SPLIT_CSV.is_file())
print("CALIB_MANIFEST :", CALIB_MANIFEST, "| ada:", CALIB_MANIFEST.is_file())

SPLIT_SHA256 = hashlib.sha256(SPLIT_CSV.read_bytes()).hexdigest()
print("SHA256 split.csv:", SPLIT_SHA256)


In [ ]:
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def count_split(split):
    d = DATA_ROOT / split
    out = {}
    if d.is_dir():
        for c in sorted(x.name for x in d.iterdir() if x.is_dir()):
            out[c] = sum(1 for f in (d / c).iterdir() if f.suffix.lower() in IMG_EXT)
    return out

counts = {s: count_split(s) for s in ("train", "val", "test")}
for s in ("train", "val", "test"):
    print(f"{s:<6} {counts[s]}  total={sum(counts[s].values())}")

if sum(counts["train"].values()) == 0:
    raise SystemExit("BERHENTI: folder train kosong.")
if sum(counts["val"].values()) == 0:
    raise SystemExit("BERHENTI: folder val tidak ditemukan / kosong. Jangan lanjut training.")
if set(counts["train"]) != set(counts["val"]) or set(counts["train"]) != set(counts["test"]):
    raise SystemExit(f"BERHENTI: daftar kelas tidak konsisten: { {k: sorted(v) for k, v in counts.items()} }")
print("OK: train/val/test lengkap dan kelas konsisten")


## 3. Training

YOLOv8n-cls pretrained, early stopping berbasis val (`patience=10`).
Augmentasi ringan: `hsv_h=0`, `hsv_s=0`, `hsv_v<=0.2`.


In [ ]:
from ultralytics import YOLO

SEED = 42
EPOCHS = 100
IMGSZ = 224
BATCH = 32
PATIENCE = 10

WORK = Path("/kaggle/working")
CFG = dict(
    data=str(DATA_ROOT), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
    patience=PATIENCE, pretrained=True, deterministic=True, workers=2,
    project=str(WORK / "runs"), name="roast_exp002_yolov8n_cls", exist_ok=True,
    fliplr=0.5, flipud=0.0, degrees=10.0, translate=0.1, scale=0.1,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.2,
)
print(json.dumps(CFG, indent=2, default=str))

model = YOLO("yolov8n-cls.pt")
results = model.train(**CFG)
RUN_DIR = Path(results.save_dir)
print("Run dir:", RUN_DIR)


## 4. Evaluasi val (seleksi model / early stopping)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

best_pt = RUN_DIR / "weights" / "best.pt"
best = YOLO(str(best_pt))

def split_files(split):
    # predict() tidak menerima path folder; kirim daftar file eksplisit.
    d = DATA_ROOT / split
    return sorted(str(f) for f in d.rglob("*") if f.suffix.lower() in IMG_EXT)

def predict_split(m, split):
    y_true, y_pred = [], []
    for r in m.predict(source=split_files(split), stream=True, imgsz=IMGSZ, verbose=False):
        y_true.append(Path(r.path).parent.name)
        y_pred.append(m.names[r.probs.top1])
    return y_true, y_pred

def report(m, split, tag):
    y_true, y_pred = predict_split(m, split)
    labels = sorted(set(y_true) | set(y_pred))
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    print(f"[{tag}] n={len(y_true)} acc={acc:.4f} macroF1={mf1:.4f}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=4))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print("confusion matrix (baris=true, kolom=pred), labels:", labels)
    print(cm)
    return {"acc": acc, "macro_f1": mf1, "cm": cm, "labels": labels}

VAL = report(best, "val", "VAL float")
vm = best.val(data=str(DATA_ROOT), split="val")
print("ultralytics val top1:", round(float(vm.top1), 4))


## 5. Evaluasi test (SEKALI di akhir)

Fokus pembacaan: pasangan **Light↔Medium** dan **Medium↔Dark**.


In [ ]:
TEST = report(best, "test", "TEST float")
tm = best.val(data=str(DATA_ROOT), split="test")
print("ultralytics test top1:", round(float(tm.top1), 4))

idx = {l: i for i, l in enumerate(TEST["labels"])}
for a, b in [("Light", "Medium"), ("Medium", "Dark")]:
    print(f"{a}->{b}: {TEST['cm'][idx[a], idx[b]]} | {b}->{a}: {TEST['cm'][idx[b], idx[a]]}")


## 6. Export INT8 + verifikasi kalibrasi

Kalibrasi memakai `calib/` yang **seluruhnya berasal dari split train**.
Sebelum export, asal-usul tiap gambar diverifikasi lewat `calib_manifest.csv`.


In [ ]:
try:
    tflite_path = best.export(format="tflite", int8=True, imgsz=IMGSZ, data=str(CALIB), split="train")
except TypeError as e:
    print("split= tidak didukung; pakai default (val). Isi calib/val juga dari train:", e)
    tflite_path = best.export(format="tflite", int8=True, imgsz=IMGSZ, data=str(CALIB))
print("TFLite INT8:", tflite_path)


## 7. Evaluasi INT8 pada test + selisih vs float


In [ ]:
OUT = WORK / "roast_exp002_artifacts"
OUT.mkdir(exist_ok=True)
shutil.copy2(best_pt, OUT / "best.pt")
shutil.copy2(tflite_path, OUT / "best_int8.tflite")
print("Artefak disimpan lebih dulu:", sorted(p.name for p in OUT.iterdir()))

acc8 = mf18 = None
int8_error = ""
try:
    int8_model = YOLO(str(tflite_path))
    y_true, y_pred = [], []
    for r in int8_model.predict(source=split_files("test"), stream=True, imgsz=IMGSZ, verbose=False):
        y_true.append(Path(r.path).parent.name)
        y_pred.append(int8_model.names[r.probs.top1])
    labels = sorted(set(y_true) | set(y_pred))
    acc8 = float(accuracy_score(y_true, y_pred))
    mf18 = float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0))
    print(f"[TEST int8] n={len(y_true)} acc={acc8:.4f} macroF1={mf18:.4f}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=4))
    print(f"Selisih float - int8: acc {TEST['acc'] - acc8:+.4f} | macroF1 {TEST['macro_f1'] - mf18:+.4f}")
except Exception as e:
    int8_error = f"{type(e).__name__}: {e}"
    print("INT8 inference gagal:", int8_error)
    print("Artefak float & .tflite tetap tersimpan; metrik INT8 ditandai tidak tersedia.")

meta = {
    "experiment_id": "EXP-ROAST-002",
    "seed": SEED, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "patience": PATIENCE,
    "ultralytics": ultralytics.__version__, "torch": torch.__version__,
    "python": sys.version.split()[0],
    "split_sha256": SPLIT_SHA256,
    "counts": counts,
    "val_acc": float(VAL["acc"]), "val_macro_f1": float(VAL["macro_f1"]),
    "test_acc": float(TEST["acc"]), "test_macro_f1": float(TEST["macro_f1"]),
    "int8_test_acc": acc8, "int8_test_macro_f1": mf18, "int8_error": int8_error,
    "hparams": CFG,
}
(OUT / "run_meta.json").write_text(json.dumps(meta, indent=2, default=str))
print(json.dumps(meta, indent=2, default=str))


## 8. Baris untuk `experiment_registry.csv`


In [ ]:
delta = None if acc8 is None else round(float(TEST["acc"]) - acc8, 4)
row = {
    "experiment_id": "EXP-ROAST-002",
    "status": "DONE",
    "model_task": "roast",
    "architecture": "yolov8n-cls",
    "datasets": "DS003-dedup",
    "classes": "Green,Light,Medium,Dark",
    "split": "70/15/15 (group-aware, split.csv)",
    "augmentation": "fliplr0.5 + degrees<=10 + translate0.1 + scale0.1 + hsv_v<=0.2",
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "acc_top1": round(float(VAL["acc"]), 4),
    "macro_f1": round(float(VAL["macro_f1"]), 4),
    "test_acc_top1": round(float(TEST["acc"]), 4),
    "test_macro_f1": round(float(TEST["macro_f1"]), 4),
    "confusion_notes": "fokus Light<->Medium, Medium<->Dark",
    "model_path": "ml/models/roast_exp002_int8.tflite",
    "git_commit": "",
    "notes": f"ultralytics {ultralytics.__version__}, torch {torch.__version__}, seed {SEED}, split.csv sha256 {SPLIT_SHA256[:12]}; INT8 test acc {acc8} (delta {delta}); int8_error={int8_error!r}",
}
print(pd.Series(row).to_string())
